# Morebench Theory Cleaning

This notebook inspects the raw `morebench_theory.csv` file, visualizes the main categorical fields,
and standardizes it into the repo's processed prompt/evaluation format.


In [ ]:
from __future__ import annotations

from pathlib import Path
from ast import literal_eval
import csv
import hashlib
import json
import re
import shutil

import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pandas as pd
except Exception as exc:
    raise RuntimeError("pandas is required to use this cleaning notebook.") from exc

from IPython.display import display



sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate
    return Path.cwd().resolve()


ROOT = find_project_root()
DATA_ROOT = ROOT / "Data" if (ROOT / "Data").exists() else ROOT / "data"
RAW_DIR = DATA_ROOT / "raw/morebench/morebench_theory.csv"
OUT_DIR = DATA_ROOT / "processed" / "morebench_theory"
SAVE_OUTPUTS = False

print("Project root:", ROOT)
print("Raw dir:", RAW_DIR)
print("Output dir:", OUT_DIR)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def normalize_for_csv(value):
    if value is None:
        return ""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value


def write_jsonl(path: Path, rows) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def write_csv(path: Path, rows, fieldnames=None) -> None:
    ensure_dir(path.parent)
    if not rows:
        return
    if fieldnames is None:
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    seen.add(key)
                    fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: normalize_for_csv(row.get(k)) for k in fieldnames})


def plot_count(series, title: str, top_n: int = 15):
    counts = series.fillna("<missing>").astype(str).value_counts().head(top_n)
    if counts.empty:
        print(f"No values available for {title}")
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(x=counts.index, y=counts.values, ax=ax, color="#4C72B0")
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("count")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def plot_text_length(series, title: str):
    lengths = series.fillna("").astype(str).str.len()
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(lengths, bins=30, ax=ax, color="#55A868")
    ax.set_title(title)
    ax.set_xlabel("characters")
    plt.tight_layout()
    plt.show()


In [ ]:
raw_df = pd.read_csv(RAW_DIR)
print("Raw shape:", raw_df.shape)
display(raw_df.head())
display(pd.DataFrame({"column": raw_df.columns, "missing": raw_df.isna().sum().values}).sort_values("missing", ascending=False))


In [ ]:
for column in ["DILEMMA_SOURCE", "DILEMMA_TYPE", "THEORY", "ROLE_DOMAIN", "CONTEXT"]:
    if column in raw_df.columns:
        plot_count(raw_df[column].astype(str), f"{column} distribution")

plot_text_length(raw_df["DILEMMA"].fillna("").astype(str), "DILEMMA length distribution")


In [ ]:
cleaned_df = pd.DataFrame({
    "text": raw_df["DILEMMA"].fillna("").astype(str).str.strip(),
    "label": "",
    "dataset": "morebench_theory",
    "task": raw_df.get("DILEMMA_TYPE", pd.Series(["prompt_eval"] * len(raw_df))).fillna("prompt_eval").astype(str),
    "split": "",
    "source_file": "morebench_theory.csv",
})
metadata_cols = [c for c in raw_df.columns if c != "DILEMMA"]
cleaned_df["metadata"] = raw_df[metadata_cols].to_dict(orient="records")

print("Cleaned shape:", cleaned_df.shape)
display(cleaned_df.head())

# Optional rubric parsing preview for spot checks.
if "RUBRIC" in raw_df.columns:
    rubric_lengths = raw_df["RUBRIC"].fillna("").astype(str).str.len()
    plot_text_length(rubric_lengths.astype(str), "Rubric string length distribution")


In [ ]:
if SAVE_OUTPUTS:
    write_jsonl(OUT_DIR / "morebench_theory.jsonl", cleaned_df.to_dict(orient="records"))
    write_csv(OUT_DIR / "morebench_theory.csv", cleaned_df.to_dict(orient="records"))
    print("Wrote cleaned Morebench Theory outputs to", OUT_DIR)
else:
    print("Preview only. Set SAVE_OUTPUTS = True and rerun this cell to write cleaned files.")
